# Wildfire Risk Prediction — Data Preprocessing

## Objective

Prepare the raw wildfire dataset for machine learning by cleaning invalid
records, handling data quality issues, creating meaningful features, and
generating a model-ready dataset while avoiding data leakage.

## Pipeline Node 01 — Remove Duplicate Records

Duplicate records were identified and removed to prevent repeated observations
from biasing model training. EDA identified 13,920 exact duplicate rows.
Only complete duplicate rows were removed while preserving valid temporal
observations.

In [1]:
# Import required libraries for data preprocessing

import pandas as pd
import numpy as np

In [3]:
# Load original wildfire dataset

DATA_PATH = "../../data/raw/Wildfire_Dataset.csv"

df_processed = pd.read_csv(DATA_PATH)

df_processed.shape

(9509925, 19)

In [4]:
# Remove completely identical rows
df_processed = df_processed.drop_duplicates()

In [5]:
# Check dataset size after removing duplicates
df_processed.shape

(9496005, 19)

In [6]:
# Confirm no duplicate rows remain
df_processed.duplicated().sum()

np.int64(0)

## Pipeline Node 02 — Handle Sentinel Values

Invalid sentinel values (32767) were identified during EDA as placeholders
for unavailable weather measurements. A total of 25,725 records containing
these values were removed because the affected rows lacked valid
environmental information. This prevents models from learning incorrect
patterns from invalid numerical values.

In [7]:
# Identify columns containing sentinel value 32767

sentinel_count = (df_processed == 32767).sum()

sentinel_count[sentinel_count > 0]

pr        25725
rmax      25725
rmin      25725
sph       25725
srad      25725
tmmn      25725
tmmx      25725
vs        25725
bi        25725
fm100     25725
fm1000    25725
erc       25725
etr       25725
pet       25725
vpd       25725
dtype: int64

In [8]:
# Count rows where any feature contains sentinel value 32767

sentinel_rows = (df_processed == 32767).any(axis=1).sum()

sentinel_rows

np.int64(25725)

In [9]:
# Remove rows containing sentinel value 32767

df_processed = df_processed[
    ~(df_processed == 32767).any(axis=1)
]

In [10]:
# Check dataset size after removing sentinel records

df_processed.shape

(9470280, 19)

In [11]:
# Confirm sentinel values are removed

(df_processed == 32767).sum().sum()

np.int64(0)

## Pipeline Node 03 — Missing Value Handling

Missing value analysis was performed after removing sentinel values.
EDA showed that unavailable weather measurements were represented using
32767 rather than standard NaN values. After handling sentinel records,
no remaining missing values were detected, therefore no imputation was
required.

In [12]:
# Check remaining missing values after sentinel removal

df_processed.isnull().sum()

latitude     0
longitude    0
datetime     0
Wildfire     0
pr           0
rmax         0
rmin         0
sph          0
srad         0
tmmn         0
tmmx         0
vs           0
bi           0
fm100        0
fm1000       0
erc          0
etr          0
pet          0
vpd          0
dtype: int64

In [13]:
# Check remaining missing values after sentinel removal

df_processed.isnull().sum()

latitude     0
longitude    0
datetime     0
Wildfire     0
pr           0
rmax         0
rmin         0
sph          0
srad         0
tmmn         0
tmmx         0
vs           0
bi           0
fm100        0
fm1000       0
erc          0
etr          0
pet          0
vpd          0
dtype: int64

## Pipeline Node 04 — Date Feature Engineering

The datetime attribute was transformed into numerical temporal features
because wildfire occurrence shows seasonal variation. Year, month, and
day_of_year features were extracted to capture long-term and seasonal
patterns. The original datetime column was removed after feature extraction.

In [14]:
df_processed["datetime"].dtype

<StringDtype(storage='python', na_value=nan)>

In [16]:
# Convert datetime column into datetime format

df_processed["datetime"] = pd.to_datetime(
    df_processed["datetime"]
)

In [17]:
# Extract useful temporal features

df_processed["year"] = df_processed["datetime"].dt.year

df_processed["month"] = df_processed["datetime"].dt.month

df_processed["day_of_year"] = df_processed["datetime"].dt.dayofyear

In [18]:
df_processed[
    [
        "datetime",
        "year",
        "month",
        "day_of_year"
    ]
].head()

,datetime,year,month,day_of_year
0,2018-08-15,2018,8,227
1,2018-08-16,2018,8,228
2,2018-08-17,2018,8,229
3,2018-08-18,2018,8,230
4,2018-08-19,2018,8,231


In [19]:
# Remove original datetime column after feature extraction

df_processed = df_processed.drop(
    columns=["datetime"]
)

## Pipeline Node 05 — Target Encoding

The wildfire target variable was converted from categorical labels into
binary numerical values. "No" was mapped to 0 and "Yes" was mapped to 1
because machine learning algorithms require numerical target representations
for classification tasks.

In [21]:
# Check unique target values before encoding

df_processed["Wildfire"].unique()

<StringArray>
['No', 'Yes']
Length: 2, dtype: str

In [22]:
# Convert wildfire labels into binary values

df_processed["Wildfire"] = df_processed["Wildfire"].map(
    {
        "No": 0,
        "Yes": 1
    }
)

In [23]:
# Check encoded target values

df_processed["Wildfire"].value_counts()

Wildfire
0    8969248
1     501032
Name: count, dtype: int64

## Pipeline Node 06 — Feature Selection

The final dataset features were reviewed based on EDA findings, leakage
analysis, and feature importance considerations. The wildfire target variable
was separated from input variables, while environmental, geographical, and
temporal features were retained because they represent factors available
during real-world wildfire prediction.

In [24]:
# View final columns after preprocessing steps

df_processed.columns.tolist()

['latitude',
 'longitude',
 'Wildfire',
 'pr',
 'rmax',
 'rmin',
 'sph',
 'srad',
 'tmmn',
 'tmmx',
 'vs',
 'bi',
 'fm100',
 'fm1000',
 'erc',
 'etr',
 'pet',
 'vpd',
 'year',
 'month',
 'day_of_year']

In [25]:
# Separate input features and target variable

X = df_processed.drop(
    columns=["Wildfire"]
)

y = df_processed["Wildfire"]

In [26]:
# Check feature shape

X.shape

(9470280, 20)

In [27]:
# Check target distribution

y.value_counts()

Wildfire
0    8969248
1     501032
Name: count, dtype: int64

## Pipeline Node 07 — Time-Based Train/Test Split

A chronological train-test split was applied because wildfire prediction is a
time-dependent problem. Historical observations were used for training while
later years were reserved for testing. This prevents temporal leakage and
better represents real-world wildfire forecasting.

In [28]:
# Split data based on year to prevent temporal leakage

train_mask = X["year"] <= 2022

test_mask = X["year"] >= 2023

In [29]:
# Create time-based train and test datasets

X_train = X[train_mask]
X_test = X[test_mask]

y_train = y[train_mask]
y_test = y[test_mask]

In [30]:
# Check train and test sizes

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)

print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (8646325, 20)
X_test: (823955, 20)
y_train: (8646325,)
y_test: (823955,)


In [31]:
print(
    "Training years:",
    X_train["year"].min(),
    "-",
    X_train["year"].max()
)

print(
    "Testing years:",
    X_test["year"].min(),
    "-",
    X_test["year"].max()
)

Training years: 2013 - 2022
Testing years: 2023 - 2025


In [32]:
print("Training distribution")
print(y_train.value_counts())

print("\nTesting distribution")
print(y_test.value_counts())

Training distribution
Wildfire
0    8263683
1     382642
Name: count, dtype: int64

Testing distribution
Wildfire
0    705565
1    118390
Name: count, dtype: int64


## Pipeline Node 08 — Feature Scaling Decision

Feature ranges were analyzed to determine whether scaling was required.
Although variables contain different numerical ranges, scaling was not
applied because the selected tree-based machine learning models are not
sensitive to feature magnitude. Scaling will only be introduced if
distance-based or gradient-based models are evaluated in future experiments.

In [33]:
# Check numerical feature ranges before modelling

X_train.describe().T

,count,mean,std,min,25%,50%,75%,max
latitude,8646325.0,39.227188,5.320252,25.26027,34.74690,38.68439,43.835700,48.99873
longitude,8646325.0,-106.268602,14.497512,-124.43700,-118.27397,-110.47140,-95.495067,-67.01250
pr,8646325.0,1.739338,6.307306,0.00000,0.00000,0.00000,0.000000,528.00000
rmax,8646325.0,76.092119,20.757756,6.00000,61.60000,79.70000,95.000000,100.00000
rmin,8646325.0,33.405653,18.309855,1.00000,19.20000,30.50000,45.200000,100.00000
sph,8646325.0,0.006061,0.003362,0.00013,0.00358,0.00548,0.007740,0.02413
srad,8646325.0,221.885020,87.660999,0.00000,148.50000,228.70000,299.700000,440.60000
tmmn,8646325.0,280.242798,8.781338,230.90000,274.70000,281.00000,286.300000,308.20000
tmmx,8646325.0,294.280198,10.253864,243.30000,287.80000,295.50000,301.900000,324.30000
vs,8646325.0,3.777909,1.675694,0.30000,2.60000,3.50000,4.600000,19.60000


In [34]:
# Save final processed dataset

df_processed.to_csv(
    "../../data/processed/wildfire_processed.csv",
    index=False
)